In [19]:
# Direct Preference Optimization

import copy
import numpy as np
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [20]:
# ------------------------------------------------
# 0) Load policies
# ------------------------------------------------------------

base_policy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Trainable policy πθ
policy_new = copy.deepcopy(base_policy).to(device).eval() # For demo clarity, keep dropout off.

# Gradients still work because requires_grad=True.
for p in policy_new.parameters():
    p.requires_grad_(True)

# Frozen reference policy πref
policy_reference = copy.deepcopy(base_policy).to(device).eval()
for p in policy_reference.parameters():
    p.requires_grad_(False)

print("Loaded:", model_name, "on", device)

Loaded: sshleifer/tiny-gpt2 on cpu


In [21]:
def encode_prompt_and_response(tokenizer, prompt_text, completion_text):
    """
    Returns:
      input_ids: [1, seq_len] for prompt+completion (teacher-forced)
      prompt_len: #tokens in prompt
      comp_len:   #tokens in completion
    """
    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(prompt_text + completion_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    prompt_len = prompt_ids.shape[1]
    comp_len   = full_ids.shape[1] - prompt_len
    assert comp_len > 0, "Completion must add at least one token."
    return full_ids, prompt_len, comp_len

In [22]:
# ------------------------------------------------------------
# 2) Response log-prob helper
# ------------------------------------------------------------

def token_logprobs_for_response(model, input_ids, prompt_len):
    """
    Computes per-token log-probs for response tokens.

    For response token at absolute position pos,
    its probability comes from logits at pos - 1.

    Returns:
        response_token_ids: [T]
        response_logps:     [T]
    """
    model_device = next(model.parameters()).device
    input_ids = input_ids.to(model_device)

    out = model(input_ids, use_cache=False)

    logits = out.logits  # [1, seq_len, vocab]
    logp_all = F.log_softmax(logits, dim=-1)

    token_ids = input_ids[0]  # [seq_len]
    seq_len = token_ids.shape[0]

    response_token_ids = token_ids[prompt_len:]  # [T]

    # Positions whose logits predict each response token.
    # response token at pos uses logits at pos - 1.
    prev_pos = torch.arange(
        prompt_len - 1,
        seq_len - 1,
        device=model_device,
    )  # [T]

    lp_rows = logp_all[0, prev_pos, :]  # [T, vocab]

    response_logps = lp_rows.gather(
        dim=1,
        index=response_token_ids.unsqueeze(1),
    ).squeeze(1)  # [T]

    return response_token_ids, response_logps


In [23]:
def sequence_logprob(model, tokenizer, prompt_text, response_text):
    """
    Computes:

        log π(response | prompt)
        = sum_t log π(response_t | prompt, response_<t)

    Returns:
        logp_sum: scalar tensor
        response_token_ids: [T]
        response_logps: [T]
    """
    input_ids, prompt_len, response_len = encode_prompt_response(
        tokenizer,
        prompt_text,
        response_text,
    )

    response_token_ids, response_logps = token_logprobs_for_response(
        model,
        input_ids,
        prompt_len,
    )

    logp_sum = response_logps.sum()

    return logp_sum, response_token_ids, response_logps

In [24]:
# ------------------------------------------------------------
# 3) DPO math helpers
# ------------------------------------------------------------

def dpo_logits(
    beta,
    logp_policy_chosen,
    logp_policy_rejected,
    logp_ref_chosen,
    logp_ref_rejected,
):
    """
    z = beta * [
            (logπθ(chosen) - logπθ(rejected))
          - (logπref(chosen) - logπref(rejected))
        ]
    """
    delta_policy = logp_policy_chosen - logp_policy_rejected
    delta_ref = logp_ref_chosen - logp_ref_rejected

    z = beta * (delta_policy - delta_ref)

    return z, delta_policy, delta_ref

In [25]:
def dpo_loss_from_logits(z):
    """
    DPO loss:

        loss = -log sigmoid(z)
             = softplus(-z)
    """
    return F.softplus(-z)


In [26]:
# ------------------------------------------------------------
# 4) One DPO train step
# ------------------------------------------------------------

def dpo_train_step(
    policy_new,
    policy_reference,
    tokenizer,
    optimizer,
    prompt,
    chosen,
    rejected,
    beta,
):
    """
    One DPO gradient step for one preference pair:

        prompt
        chosen response
        rejected response

    policy_new:
        trainable policy πθ

    policy_reference:
        frozen reference policy πref
    """

    # --------------------------------------------------------
    # A) Policy log-probs with gradients
    # --------------------------------------------------------
    logp_policy_chosen, chosen_ids, chosen_logps = sequence_logprob(
        policy_new,
        tokenizer,
        prompt,
        chosen,
    )

    logp_policy_rejected, rejected_ids, rejected_logps = sequence_logprob(
        policy_new,
        tokenizer,
        prompt,
        rejected,
    )

    # --------------------------------------------------------
    # B) Reference log-probs without gradients
    # --------------------------------------------------------
    with torch.no_grad():
        logp_ref_chosen, _, _ = sequence_logprob(
            policy_reference,
            tokenizer,
            prompt,
            chosen,
        )

        logp_ref_rejected, _, _ = sequence_logprob(
            policy_reference,
            tokenizer,
            prompt,
            rejected,
        )

    # --------------------------------------------------------
    # C) DPO loss
    # --------------------------------------------------------
    z, delta_policy, delta_ref = dpo_logits(
        beta,
        logp_policy_chosen,
        logp_policy_rejected,
        logp_ref_chosen,
        logp_ref_rejected,
    )

    loss = dpo_loss_from_logits(z)

    pref_prob = torch.sigmoid(z)

    # --------------------------------------------------------
    # D) Optimizer update
    # --------------------------------------------------------
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # Return useful scalars from before the step.
    stats = {
        "logp_policy_chosen": float(logp_policy_chosen.detach().cpu()),
        "logp_policy_rejected": float(logp_policy_rejected.detach().cpu()),
        "logp_ref_chosen": float(logp_ref_chosen.detach().cpu()),
        "logp_ref_rejected": float(logp_ref_rejected.detach().cpu()),
        "delta_policy": float(delta_policy.detach().cpu()),
        "delta_ref": float(delta_ref.detach().cpu()),
        "z": float(z.detach().cpu()),
        "pref_prob": float(pref_prob.detach().cpu()),
        "loss": float(loss.detach().cpu()),
    }

    return stats

In [27]:
# ------------------------------------------------------------
# 5) Evaluation helper
# ------------------------------------------------------------

@torch.no_grad()
def dpo_eval_stats(
    policy_new,
    policy_reference,
    tokenizer,
    prompt,
    chosen,
    rejected,
    beta,
):
    """
    Computes DPO stats without updating.
    """
    logp_policy_chosen, _, _ = sequence_logprob(
        policy_new,
        tokenizer,
        prompt,
        chosen,
    )

    logp_policy_rejected, _, _ = sequence_logprob(
        policy_new,
        tokenizer,
        prompt,
        rejected,
    )

    logp_ref_chosen, _, _ = sequence_logprob(
        policy_reference,
        tokenizer,
        prompt,
        chosen,
    )

    logp_ref_rejected, _, _ = sequence_logprob(
        policy_reference,
        tokenizer,
        prompt,
        rejected,
    )

    z, delta_policy, delta_ref = dpo_logits(
        beta,
        logp_policy_chosen,
        logp_policy_rejected,
        logp_ref_chosen,
        logp_ref_rejected,
    )

    loss = dpo_loss_from_logits(z)
    pref_prob = torch.sigmoid(z)

    return {
        "logp_policy_chosen": float(logp_policy_chosen.detach().cpu()),
        "logp_policy_rejected": float(logp_policy_rejected.detach().cpu()),
        "logp_ref_chosen": float(logp_ref_chosen.detach().cpu()),
        "logp_ref_rejected": float(logp_ref_rejected.detach().cpu()),
        "delta_policy": float(delta_policy.detach().cpu()),
        "delta_ref": float(delta_ref.detach().cpu()),
        "z": float(z.detach().cpu()),
        "pref_prob": float(pref_prob.detach().cpu()),
        "loss": float(loss.detach().cpu()),
    }


In [28]:
# ------------------------------------------------------------
# 6) DPO settings
# ------------------------------------------------------------

beta = 0.1
lr = 1e-5

optimizer = torch.optim.Adam(
    policy_new.parameters(),
    lr=lr,
)


# ------------------------------------------------------------
# 7) Two preference examples, batch size = 1
# ------------------------------------------------------------

examples = [
    {
        "prompt": "The capital of France is",
        "chosen": " Paris.",
        "rejected": " London.",
    },
    {
        "prompt": "2 + 2 =",
        "chosen": " 4.",
        "rejected": " 5.",
    },
]


# ------------------------------------------------------------
# 8) Main DPO flow
# ------------------------------------------------------------

for example_idx, ex in enumerate(examples, start=1):

    print("\n" + "=" * 80)
    print(f"EXAMPLE {example_idx} / BATCH SIZE = 1")
    print("=" * 80)

    prompt = ex["prompt"]
    chosen = ex["chosen"]
    rejected = ex["rejected"]

    print("Prompt:  ", repr(prompt))
    print("Chosen:  ", repr(chosen))
    print("Rejected:", repr(rejected))

    print("\n[1] Before DPO update")

    before = dpo_eval_stats(
        policy_new,
        policy_reference,
        tokenizer,
        prompt,
        chosen,
        rejected,
        beta,
    )

    print("    delta_policy:", before["delta_policy"])
    print("    delta_ref:   ", before["delta_ref"])
    print("    z:           ", before["z"])
    print("    pref_prob:   ", before["pref_prob"])
    print("    loss:        ", before["loss"])

    print("\n[2] DPO train step")

    train_stats = dpo_train_step(
        policy_new,
        policy_reference,
        tokenizer,
        optimizer,
        prompt,
        chosen,
        rejected,
        beta,
    )

    print("    loss used for update:", train_stats["loss"])

    print("\n[3] After DPO update")

    after = dpo_eval_stats(
        policy_new,
        policy_reference,
        tokenizer,
        prompt,
        chosen,
        rejected,
        beta,
    )

    print("    delta_policy:", after["delta_policy"])
    print("    delta_ref:   ", after["delta_ref"])
    print("    z:           ", after["z"])
    print("    pref_prob:   ", after["pref_prob"])
    print("    loss:        ", after["loss"])

    print("\nInterpretation:")
    print("    DPO should push delta_policy upward relative to delta_ref.")
    print("    That means policy_new should prefer chosen over rejected more strongly.")


EXAMPLE 1 / BATCH SIZE = 1
Prompt:   'The capital of France is'
Chosen:   ' Paris.'
Rejected: ' London.'

[1] Before DPO update
    delta_policy: -0.015232086181640625
    delta_ref:    -0.015232086181640625
    z:            0.0
    pref_prob:    0.5
    loss:         0.6931471824645996

[2] DPO train step
    loss used for update: 0.6931471824645996

[3] After DPO update
    delta_policy: -0.01390838623046875
    delta_ref:    -0.015232086181640625
    z:            0.00013236999802757055
    pref_prob:    0.5000330805778503
    loss:         0.6930809617042542

Interpretation:
    DPO should push delta_policy upward relative to delta_ref.
    That means policy_new should prefer chosen over rejected more strongly.

EXAMPLE 2 / BATCH SIZE = 1
Prompt:   '2 + 2 ='
Chosen:   ' 4.'
Rejected: ' 5.'

[1] Before DPO update
    delta_policy: -0.015417098999023438
    delta_ref:    -0.01541900634765625
    z:            1.9073486612342094e-07
    pref_prob:    0.5000000596046448
    loss:    